# Multi-trip distance-along-shape exploration

Projects every Culver City vehicle position recorded against a single shape on a single service date onto that shape, smooths each trip's trajectory, and overlays all trips on diagnostic distance- and speed-over-elapsed-time plots. Per-trip signal/stop delays are not computed here yet.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from constants import CA_NAD83_Albers, CULVER_CITY_FEED_KEY, MAX_SHAPE_JUMP_M, MAX_SNAP_DISTANCE_M, SERVICE_DATE, SHAPE_KEY_TO_SHAPE_ID_MAP
from _data_loaders import get_culver_city_vehicle_positions, get_selected_shapes, get_traffic_signals
from match_shapes_vp import project_vp_on_shape, project_points_on_shape
from smooth_trajectory import smooth_distances_per_trip, compute_speeds_per_trip
from plot_trip import plot_distance_over_time, plot_speed_over_time

## Configuration

Pick the shape (by ROUTE_ID key) and the service date to analyze. The vehicle-position folder spans service dates 2026-01-31 to 2026-02-16; set `SERVICE_DATE` to any date in that range (it defaults to the value in `constants.py`).

In [ ]:
SHAPE_KEY = "105"
SHAPE_ID = SHAPE_KEY_TO_SHAPE_ID_MAP[SHAPE_KEY]

# Service date to analyze. Override the constants.py default with any date in the
# vehicle-position folder (2026-01-31 .. 2026-02-16).
SERVICE_DATE = "2026-02-03"

STOP_PLOT_SNAP_DISTANCE_M = 5
HIGHLIGHT_DELAY_THRESHOLD_S = 1.0

print(f"Shape: {SHAPE_KEY} -> {SHAPE_ID}")
print(f"Service date: {SERVICE_DATE}")

## Data loading

In [ ]:
vp = get_culver_city_vehicle_positions([SHAPE_KEY], SERVICE_DATE)

### Trips per service day

Count unique trips per day in the loaded VP for the selected shape.

In [ ]:
vp.groupby(vp["event_time_datetime"].dt.date)["TRIP_KEY"].nunique().rename("n_trips").to_frame()

In [ ]:
shapes = get_selected_shapes(SERVICE_DATE, CULVER_CITY_FEED_KEY, [SHAPE_ID])

In [ ]:
signals = get_traffic_signals()

In [ ]:
stops = gpd.read_file(f"data/stops_{SHAPE_ID}.geojson").to_crs(CA_NAD83_Albers)

## Trip smoothing

In [ ]:
vp["distance_along_shape"] = project_vp_on_shape(vp, shapes, SHAPE_KEY_TO_SHAPE_ID_MAP, max_snap_distance=MAX_SNAP_DISTANCE_M, max_shape_jump=MAX_SHAPE_JUMP_M)
vp_smoothed = smooth_distances_per_trip(vp, vp["distance_along_shape"], freq_seconds=1.0)
vp_speeds = compute_speeds_per_trip(vp_smoothed, freq_seconds=1.0)

## Fixed-point projection

Signal and stop projections are shape-invariant, so they only need to be computed once for all trips.

In [ ]:
shape_geom = shapes.set_index("shape_id")["geometry"]
trip_shape = shape_geom[[SHAPE_ID]]

signal_distances = project_points_on_shape(signals, trip_shape, MAX_SNAP_DISTANCE_M)
nearside_stop_distances = project_points_on_shape(stops[stops["nearside"] == True], trip_shape, STOP_PLOT_SNAP_DISTANCE_M)
farside_stop_distances = project_points_on_shape(stops[stops["nearside"] != True], trip_shape, STOP_PLOT_SNAP_DISTANCE_M)

## Split per trip

Group the projected/smoothed VP into a list per trip for plotting. Trips without smoothed samples are skipped.

In [ ]:
trip_keys = sorted(vp_smoothed["TRIP_KEY"].unique())

trips = []
trips_smoothed = []
trips_speeds = []

for trip_key in trip_keys:
    trip = vp[vp["TRIP_KEY"] == trip_key].sort_values("event_time_datetime")
    trip_smoothed = vp_smoothed[vp_smoothed["TRIP_KEY"] == trip_key].sort_values("event_time_datetime")
    trip_speeds = vp_speeds[vp_speeds["TRIP_KEY"] == trip_key].sort_values("event_time_datetime")
    if trip_smoothed.empty or trip_speeds.empty:
        continue
    trips.append(trip)
    trips_smoothed.append(trip_smoothed)
    trips_speeds.append(trip_speeds)

print(f"Plotting {len(trips)} trips for shape {SHAPE_ID}")

## Plots

### Distance along shape over elapsed time

In [ ]:
fig_dist, ax_dist = plot_distance_over_time(
    trips=trips,
    signal_distances=signal_distances,
    nearside_stop_distances=nearside_stop_distances,
    farside_stop_distances=farside_stop_distances,
    trips_smoothed=trips_smoothed,
    highlight_delay_threshold_s=HIGHLIGHT_DELAY_THRESHOLD_S,
)
fig_dist.savefig(f"all_trips_distance_{SHAPE_ID}.png", dpi=150)
plt.show()

### Speed over elapsed time

In [ ]:
fig_speed, ax_speed = plot_speed_over_time(
    trips_speeds=trips_speeds,
    highlight_delay_threshold_s=HIGHLIGHT_DELAY_THRESHOLD_S,
)
fig_speed.savefig(f"all_trips_speed_{SHAPE_ID}.png", dpi=150)
plt.show()